# TinyLlama LoRA to GGUF on Colab

Use this notebook from a fresh Colab GPU runtime. It trains the TinyLlama LoRA adapter, merges it, converts to GGUF in an isolated conversion environment, and exports artifacts to Google Drive.

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from datetime import datetime, timezone

REPO_URL = 'https://github.com/ashioyajotham/weather_forecasting_lora.git'
os.environ['REPO_URL'] = REPO_URL
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_ROOT = '/content/drive/MyDrive/weather_forecasting_lora_runs'
RUN_DIR = f'{DRIVE_ROOT}/tinyllama_gguf_{RUN_ID}'
os.environ['RUN_DIR'] = RUN_DIR
os.makedirs(RUN_DIR, exist_ok=True)
print('RUN_DIR=', RUN_DIR)

In [ ]:
%%bash
set -euo pipefail
rm -rf /content/weather_forecasting_lora
git clone "$REPO_URL" /content/weather_forecasting_lora
cd /content/weather_forecasting_lora
git rev-parse HEAD

In [ ]:
%cd /content/weather_forecasting_lora
import os

os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WEATHER_LORA_REPORT_TO'] = 'none'

# First retry should be 200. Increase to 1000 or full data after a clean run.
os.environ['WEATHER_LORA_MAX_SAMPLES'] = os.environ.get('WEATHER_LORA_MAX_SAMPLES', '200')
print('WEATHER_LORA_MAX_SAMPLES=', os.environ['WEATHER_LORA_MAX_SAMPLES'])

In [ ]:
%%bash
set -euo pipefail
python -m pip install -U pip
python -m pip install -r requirements-colab.txt
python -m pip install bitsandbytes

In [ ]:
import torch
import torchvision
import transformers
import peft
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer
from peft import PeftModel

print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('transformers:', transformers.__version__)
print('peft:', peft.__version__)

assert torch.cuda.is_available(), 'CUDA is not available; switch Colab runtime to GPU before training.'

In [ ]:
%%bash
set -euo pipefail
python scripts/prepare_data.py --min-train 240 --min-val 30 --min-test 30 --synthetic-if-missing
python scripts/smoke_check.py
python scripts/eval_smoke.py
python scripts/generation_quality_eval.py --write-report "$RUN_DIR/generation_quality_baseline.json"

In [ ]:
%%bash
set -euo pipefail
python train_lora_peft.py 2>&1 | tee "$RUN_DIR/train_lora_peft.log"
test -s models/weather-lora-peft/lora_adapter/adapter_model.safetensors

In [ ]:
%%bash
set -euo pipefail
python merge_lora.py 2>&1 | tee "$RUN_DIR/merge_lora.log"
test -s models/weather-merged/config.json

## GGUF conversion

Conversion runs in a separate virtual environment so llama.cpp conversion requirements cannot replace the CUDA training stack.

In [ ]:
%%bash
set -euo pipefail
test -s models/weather-merged/config.json
rm -rf /content/llama.cpp /content/gguf-venv
git clone https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
python -m venv /content/gguf-venv
/content/gguf-venv/bin/python -m pip install -U pip
/content/gguf-venv/bin/python -m pip install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
mkdir -p models/gguf
/content/gguf-venv/bin/python /content/llama.cpp/convert_hf_to_gguf.py models/weather-merged --outfile models/gguf/weather-tinyllama.gguf --outtype f16 2>&1 | tee "$RUN_DIR/convert_gguf.log"
test -s models/gguf/weather-tinyllama.gguf

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

paths = {
    'adapter': Path('models/weather-lora-peft/lora_adapter'),
    'merged': Path('models/weather-merged'),
    'gguf': Path('models/gguf/weather-tinyllama.gguf'),
}

def path_size(path: Path) -> int:
    if path.is_file():
        return path.stat().st_size
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file())

for name, path in paths.items():
    if not path.exists() or path_size(path) == 0:
        raise RuntimeError(f'Missing non-empty artifact: {name} -> {path}')
    target = Path(RUN_DIR) / name
    if path.is_dir():
        if target.exists():
            shutil.rmtree(target)
        shutil.copytree(path, target)
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)

manifest = {
    'run_id': RUN_ID,
    'repo_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'gpu': subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader | head -n 1"),
    'weather_lora_max_samples': os.environ.get('WEATHER_LORA_MAX_SAMPLES'),
    'artifacts': {name: {'path': str(Path(RUN_DIR) / name), 'bytes': path_size(Path(RUN_DIR) / name)} for name in paths},
}

if any(item['bytes'] <= 0 for item in manifest['artifacts'].values()):
    raise RuntimeError(f'Invalid zero-byte artifact manifest: {manifest}')

manifest_path = Path(RUN_DIR) / 'manifest_tinyllama_gguf.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## Local import after Colab

Copy the Drive `adapter`, `merged`, and `gguf` artifacts into local `models/weather-lora-peft/lora_adapter`, `models/weather-merged`, and `models/gguf/weather-tinyllama.gguf`. Then run local smoke checks and CLI/FastAPI inference before treating the run as accepted.